# Microfinance Loan Repayment Prediction

## 02 - Data Cleaning

This notebook performs data validation and cleaning of the microfinance transaction dataset before exploratory data analysis and machine learning.

### Cleaning objectives

- Remove index and identifier columns that should not be used as predictive features.
- Remove constant columns.
- Validate data types.
- Validate target values.
- Check missing values.
- Check duplicate records.
- Investigate abnormal and extreme numerical values.
- Preserve the original raw dataset.

In [2]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)

In [3]:
PROJECT_ROOT = Path.cwd().parent

RAW_DATA = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA = PROJECT_ROOT / "data" / "processed"

CSV_PATH = RAW_DATA / "Micro-credit-Data-file.csv"

df = pd.read_csv(CSV_PATH)

print("Dataset shape:", df.shape)

Dataset shape: (209593, 37)


In [4]:
df_clean = df.copy()

In [5]:
if "Unnamed: 0" in df_clean.columns:
    df_clean = df_clean.drop(columns=["Unnamed: 0"])

print(df_clean.shape)

(209593, 36)


In [6]:
if "msisdn" in df_clean.columns:
    df_clean = df_clean.drop(columns=["msisdn"])

In [7]:
constant_columns = [
    col for col in df_clean.columns
    if df_clean[col].nunique(dropna=False) <= 1
]

print("Constant columns:")
print(constant_columns)

Constant columns:
['pcircle']


In [8]:
df_clean = df_clean.drop(columns=constant_columns)

In [9]:
TARGET = "label"

print("Target values:")
print(sorted(df_clean[TARGET].unique()))

Target values:
[np.int64(0), np.int64(1)]


In [10]:
invalid_target = ~df_clean[TARGET].isin([0, 1])

print("Invalid target records:", invalid_target.sum())

Invalid target records: 0


In [11]:
missing = df_clean.isnull().sum()

print(
    "Total missing values:",
    missing.sum()
)

Total missing values: 0


In [12]:
duplicates = df_clean.duplicated().sum()

print("Duplicate rows:", duplicates)

Duplicate rows: 31


In [13]:
df_clean.dtypes

label                     int64
aon                     float64
daily_decr30            float64
daily_decr90            float64
rental30                float64
rental90                float64
last_rech_date_ma       float64
last_rech_date_da       float64
last_rech_amt_ma          int64
cnt_ma_rech30             int64
fr_ma_rech30            float64
sumamnt_ma_rech30       float64
medianamnt_ma_rech30    float64
medianmarechprebal30    float64
cnt_ma_rech90             int64
fr_ma_rech90              int64
sumamnt_ma_rech90         int64
medianamnt_ma_rech90    float64
medianmarechprebal90    float64
cnt_da_rech30           float64
fr_da_rech30            float64
cnt_da_rech90             int64
fr_da_rech90              int64
cnt_loans30               int64
amnt_loans30              int64
maxamnt_loans30         float64
medianamnt_loans30      float64
cnt_loans90             float64
amnt_loans90              int64
maxamnt_loans90           int64
medianamnt_loans90      float64
payback3

In [14]:
df_clean["pdate"] = pd.to_datetime(
    df_clean["pdate"],
    errors="coerce"
)

print(df_clean["pdate"].dtype)

datetime64[us]


In [15]:
print("Invalid dates:", df_clean["pdate"].isna().sum())

Invalid dates: 0


In [16]:
print("Minimum date:", df_clean["pdate"].min())
print("Maximum date:", df_clean["pdate"].max())

Minimum date: 2016-06-01 00:00:00
Maximum date: 2016-08-21 00:00:00


In [17]:
numeric_cols = df_clean.select_dtypes(
    include=["number"]
).columns

df_clean[numeric_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
label,209593.0,0.875177,0.330519,0.000000,1.000,1.000000,1.00,1.000000
aon,209593.0,8112.343445,75696.082531,-48.000000,246.000,527.000000,982.00,999860.755168
daily_decr30,209593.0,5381.402289,9220.623400,-93.012667,42.440,1469.175667,7244.00,265926.000000
daily_decr90,209593.0,6082.515068,10918.812767,-93.012667,42.692,1500.000000,7802.79,320630.000000
rental30,209593.0,2692.581910,4308.586781,-23737.140000,280.420,1083.570000,3356.94,198926.110000
rental90,209593.0,3483.406534,5770.461279,-24720.580000,300.260,1334.000000,4201.79,200148.110000
last_rech_date_ma,209593.0,3755.847800,53905.892230,-29.000000,1.000,3.000000,7.00,998650.377733
last_rech_date_da,209593.0,3712.202921,53374.833430,-29.000000,0.000,0.000000,0.00,999171.809410
last_rech_amt_ma,209593.0,2064.452797,2370.786034,0.000000,770.000,1539.000000,2309.00,55000.000000
cnt_ma_rech30,209593.0,3.978057,4.256090,0.000000,1.000,3.000000,5.00,203.000000


In [18]:
negative_summary = pd.DataFrame({
    "Negative_Count": [
        (df_clean[col] < 0).sum()
        for col in numeric_cols
    ]
}, index=numeric_cols)

negative_summary = negative_summary[
    negative_summary["Negative_Count"] > 0
]

negative_summary

,Negative_Count
aon,1539
daily_decr30,1839
daily_decr90,1839
rental30,5628
rental90,5929
last_rech_date_ma,1315
last_rech_date_da,14
medianmarechprebal30,1646
medianmarechprebal90,1730


In [19]:
percentiles = df_clean[numeric_cols].quantile(
    [0.01, 0.25, 0.50, 0.75, 0.99]
).T

percentiles.columns = [
    "1%",
    "25%",
    "50%",
    "75%",
    "99%"
]

percentiles

,1%,25%,50%,75%,99%
label,0.0000,1.000,1.000000,1.00,1.000000
aon,55.0000,246.000,527.000000,982.00,2420.080000
daily_decr30,0.0000,42.440,1469.175667,7244.00,41730.440000
daily_decr90,0.0000,42.692,1500.000000,7802.79,49967.383600
rental30,-299.9304,280.420,1083.570000,3356.94,19461.376400
rental90,-349.5848,300.260,1334.000000,4201.79,26997.968000
last_rech_date_ma,0.0000,1.000,3.000000,7.00,57.000000
last_rech_date_da,0.0000,0.000,0.000000,0.00,56.000000
last_rech_amt_ma,0.0000,770.000,1539.000000,2309.00,10000.000000
cnt_ma_rech30,0.0000,1.000,3.000000,5.00,20.000000


In [20]:
# Investigate extreme values

check_columns = [
    "aon",
    "daily_decr30",
    "daily_decr90",
    "rental30",
    "rental90",
    "last_rech_date_ma",
    "last_rech_date_da",
    "fr_ma_rech30",
    "medianmarechprebal30",
    "cnt_da_rech30",
    "fr_da_rech30",
    "maxamnt_loans30",
    "cnt_loans90"
]

for col in check_columns:
    print("\n" + "=" * 70)
    print(f"Column: {col}")
    print("Minimum:", df_clean[col].min())
    print("Maximum:", df_clean[col].max())
    print("Top 10 largest values:")
    print(df_clean[col].nlargest(10).to_list())


Column: aon
Minimum: -48.0
Maximum: 999860.755167902
Top 10 largest values:
[999860.755167902, 999520.609388128, 999391.307123005, 999215.59705399, 999190.832138993, 998995.927511714, 998013.356234878, 997897.53172081, 997797.27961868, 997608.519508503]

Column: daily_decr30
Minimum: -93.0126666666667
Maximum: 265926.0
Top 10 largest values:
[265926.0, 212364.0, 212202.0, 185313.0, 183850.0, 173834.0, 169237.902666667, 167924.5, 167693.0, 167089.0]

Column: daily_decr90
Minimum: -93.0126666666667
Maximum: 320630.0
Top 10 largest values:
[320630.0, 259525.0, 254657.13, 244906.76, 231228.81, 217542.5, 210520.82, 207595.96, 194024.49, 193841.9]

Column: rental30
Minimum: -23737.14
Maximum: 198926.11
Top 10 largest values:
[198926.11, 147045.42, 124355.64, 107663.98, 104277.59, 104265.76, 98007.55, 94161.92, 82461.12, 81478.24]

Column: rental90
Minimum: -24720.58
Maximum: 200148.11
Top 10 largest values:
[200148.11, 172084.42, 169392.92, 167673.7, 157342.91, 153391.19, 144788.71, 126887.

In [21]:
# Check suspicious very large values

suspicious_threshold = 900000

suspicious_counts = {}

for col in numeric_cols:
    count = (df_clean[col].abs() >= suspicious_threshold).sum()
    
    if count > 0:
        suspicious_counts[col] = count

pd.Series(
    suspicious_counts,
    name="Suspicious_Count"
).sort_values(ascending=False)

aon                     406
medianmarechprebal30    222
last_rech_date_ma       218
last_rech_date_da       206
fr_da_rech30            201
fr_ma_rech30            193
sumamnt_ma_rech90         1
Name: Suspicious_Count, dtype: int64

In [22]:
for col in suspicious_counts.keys():
    print("\n" + "=" * 70)
    print(f"{col}")
    
    values = df_clean.loc[
        df_clean[col].abs() >= suspicious_threshold,
        col
    ].value_counts().head(15)
    
    print(values)


aon
aon
929670.835962    1
946234.231931    1
900991.102797    1
944133.631536    1
954313.001479    1
931789.404363    1
928493.559943    1
965019.931202    1
973610.205576    1
962216.084474    1
910990.614910    1
923787.519103    1
995119.092520    1
973628.493142    1
933141.397196    1
Name: count, dtype: int64

last_rech_date_ma
last_rech_date_ma
942339.085159    1
942783.444771    1
995749.962400    1
940786.194871    1
968306.984170    1
917314.397986    1
989570.382400    1
961851.046886    1
981248.171302    1
911607.378279    1
989438.291756    1
968429.340515    1
984943.871619    1
933237.113408    1
953684.656648    1
Name: count, dtype: int64

last_rech_date_da
last_rech_date_da
995586.030185    1
906659.029075    1
995848.999708    1
966148.149222    1
994015.222648    1
903481.430607    1
923275.847337    1
955954.479403    1
978462.602478    1
976663.801121    1
953794.303467    1
925288.839615    1
948381.002876    1
987091.889489    1
983069.038950    1
Name: coun

In [23]:
negative_details = []

for col in numeric_cols:
    neg_count = (df_clean[col] < 0).sum()
    
    if neg_count > 0:
        negative_details.append({
            "column": col,
            "negative_count": neg_count,
            "negative_percentage": round(
                neg_count / len(df_clean) * 100,
                4
            ),
            "minimum": df_clean[col].min()
        })

negative_details = pd.DataFrame(negative_details)

negative_details

,column,negative_count,negative_percentage,minimum
0,aon,1539,0.7343,-48.000000
1,daily_decr30,1839,0.8774,-93.012667
2,daily_decr90,1839,0.8774,-93.012667
3,rental30,5628,2.6852,-23737.140000
4,rental90,5929,2.8288,-24720.580000
5,last_rech_date_ma,1315,0.6274,-29.000000
6,last_rech_date_da,14,0.0067,-29.000000
7,medianmarechprebal30,1646,0.7853,-200.000000
8,medianmarechprebal90,1730,0.8254,-200.000000


In [24]:
print(df_clean["pdate"].dtype)

print("\nDate range:")
print(df_clean["pdate"].min())
print(df_clean["pdate"].max())

print("\nInvalid dates:")
print(df_clean["pdate"].isna().sum())

datetime64[us]

Date range:
2016-06-01 00:00:00
2016-08-21 00:00:00

Invalid dates:
0


In [25]:
# Identify suspicious extreme values

suspicious_threshold = 900000

suspicious_mask = pd.Series(False, index=df_clean.index)

for col in numeric_cols:
    suspicious_mask |= df_clean[col].abs() >= suspicious_threshold

print("Rows containing at least one suspicious extreme value:",
      suspicious_mask.sum())

print("Percentage:",
      round(suspicious_mask.mean() * 100, 4), "%")

Rows containing at least one suspicious extreme value: 1442
Percentage: 0.688 %


In [26]:
df_clean = df_clean[~suspicious_mask]

In [27]:
suspicious_columns = [
    "aon",
    "last_rech_date_ma",
    "last_rech_date_da",
    "fr_ma_rech30",
    "fr_da_rech30",
    "cnt_da_rech30",
    "fr_da_rech90"
]

print("Suspicious columns:")
for col in suspicious_columns:
    print(col)

Suspicious columns:
aon
last_rech_date_ma
last_rech_date_da
fr_ma_rech30
fr_da_rech30
cnt_da_rech30
fr_da_rech90


In [28]:
# Convert clearly invalid extreme values to NaN

for col in suspicious_columns:
    mask = df_clean[col].abs() >= 900000
    
    print(
        f"{col}: {mask.sum()} suspicious values"
    )
    
    df_clean.loc[mask, col] = np.nan

aon: 0 suspicious values
last_rech_date_ma: 0 suspicious values
last_rech_date_da: 0 suspicious values
fr_ma_rech30: 0 suspicious values
fr_da_rech30: 0 suspicious values
cnt_da_rech30: 0 suspicious values
fr_da_rech90: 0 suspicious values


In [29]:
missing_after_extreme = (
    df_clean.isnull()
    .sum()
    .sort_values(ascending=False)
)

missing_after_extreme[
    missing_after_extreme > 0
]

Series([], dtype: int64)

In [30]:
for col in numeric_cols:
    if df_clean[col].isna().sum() > 0:
        df_clean[col] = df_clean[col].fillna(
            df_clean[col].median()
        )

In [31]:
print("Total missing cells:",
      df_clean.isna().sum().sum())

Total missing cells: 0


In [32]:
remaining_suspicious = {}

for col in numeric_cols:
    count = (
        df_clean[col].abs() >= suspicious_threshold
    ).sum()
    
    if count > 0:
        remaining_suspicious[col] = count

pd.Series(
    remaining_suspicious,
    name="Remaining_Suspicious_Count"
).sort_values(ascending=False)

Series([], Name: Remaining_Suspicious_Count, dtype: object)

In [33]:
print("Final shape:", df_clean.shape)
print("Total rows:", len(df_clean))
print("Total columns:", df_clean.shape[1])

Final shape: (208151, 34)
Total rows: 208151
Total columns: 34


In [34]:
print(df_clean["label"].value_counts())

print("\nTarget proportions:")
print(
    df_clean["label"]
    .value_counts(normalize=True)
    .round(4)
)

label
1    182166
0     25985
Name: count, dtype: int64

Target proportions:
label
1    0.8752
0    0.1248
Name: proportion, dtype: float64


In [35]:
# Reload the original raw dataset
df_raw = pd.read_csv(CSV_PATH)

print("Raw shape:", df_raw.shape)
print("Raw target distribution:")
print(df_raw["label"].value_counts())

Raw shape: (209593, 37)
Raw target distribution:
label
1    183431
0     26162
Name: count, dtype: int64


In [36]:
# Create a fresh cleaning dataframe
df_clean = df_raw.copy()

# Remove index column
if "Unnamed: 0" in df_clean.columns:
    df_clean = df_clean.drop(columns=["Unnamed: 0"])

# Remove customer identifier
if "msisdn" in df_clean.columns:
    df_clean = df_clean.drop(columns=["msisdn"])

# Remove constant columns
constant_columns = [
    col for col in df_clean.columns
    if df_clean[col].nunique(dropna=False) <= 1
]

print("Constant columns:", constant_columns)

df_clean = df_clean.drop(columns=constant_columns)

print("Shape after structural cleaning:", df_clean.shape)

Constant columns: ['pcircle']
Shape after structural cleaning: (209593, 34)


In [37]:
df_clean["pdate"] = pd.to_datetime(
    df_clean["pdate"],
    errors="coerce"
)

print("Date dtype:", df_clean["pdate"].dtype)
print("Invalid dates:", df_clean["pdate"].isna().sum())

Date dtype: datetime64[us]
Invalid dates: 0


In [38]:
suspicious_columns = [
    "aon",
    "last_rech_date_ma",
    "last_rech_date_da",
    "fr_ma_rech30",
    "fr_da_rech30",
    "cnt_da_rech30",
    "fr_da_rech90"
]

for col in suspicious_columns:
    mask = df_clean[col].abs() >= 900000
    
    print(f"{col}: {mask.sum()} suspicious values")
    
    # Replace suspicious values with NaN
    df_clean.loc[mask, col] = np.nan

aon: 406 suspicious values
last_rech_date_ma: 218 suspicious values
last_rech_date_da: 206 suspicious values
fr_ma_rech30: 193 suspicious values
fr_da_rech30: 201 suspicious values
cnt_da_rech30: 0 suspicious values
fr_da_rech90: 0 suspicious values


In [39]:
for col in suspicious_columns:
    if df_clean[col].isna().sum() > 0:
        median_value = df_clean[col].median()
        df_clean[col] = df_clean[col].fillna(median_value)

In [40]:
print("Missing cells:", df_clean.isna().sum().sum())

Missing cells: 0


In [41]:
print("Final cleaning shape:", df_clean.shape)
print("Rows:", len(df_clean))
print("Columns:", len(df_clean.columns))

Final cleaning shape: (209593, 34)
Rows: 209593
Columns: 34


In [42]:
print("Target distribution:")
print(df_clean["label"].value_counts())

print("\nTarget proportions:")
print(
    df_clean["label"]
    .value_counts(normalize=True)
    .round(4)
)

Target distribution:
label
1    183431
0     26162
Name: count, dtype: int64

Target proportions:
label
1    0.8752
0    0.1248
Name: proportion, dtype: float64


In [43]:
print("Duplicate rows:", df_clean.duplicated().sum())

Duplicate rows: 31


In [44]:
remaining_suspicious = {}

numeric_cols_clean = df_clean.select_dtypes(
    include=["number"]
).columns

for col in numeric_cols_clean:
    count = (
        df_clean[col].abs() >= 900000
    ).sum()
    
    if count > 0:
        remaining_suspicious[col] = count

print(
    pd.Series(
        remaining_suspicious,
        name="Remaining_Suspicious_Count"
    )
)

medianmarechprebal30    222
sumamnt_ma_rech90         1
Name: Remaining_Suspicious_Count, dtype: int64


In [45]:
remaining_columns = [
    "medianmarechprebal30",
    "sumamnt_ma_rech90"
]

for col in remaining_columns:
    print("\n" + "=" * 70)
    print("COLUMN:", col)
    
    suspicious = df_clean.loc[
        df_clean[col].abs() >= 900000,
        col
    ]
    
    print("Suspicious count:", len(suspicious))
    print("\nValues:")
    print(suspicious.to_list())


COLUMN: medianmarechprebal30
Suspicious count: 222

Values:
[977399.19507876, 912257.064483128, 936286.288662814, 945714.040775783, 993389.199720696, 989909.946452826, 914101.316710003, 944444.017717615, 979177.235858515, 942989.731440321, 907652.472145855, 926141.349365935, 938906.900468282, 942203.037673607, 908647.81755954, 986044.694902375, 926147.627644241, 915853.804675862, 936147.849773988, 983869.041898288, 906080.920132808, 962416.638736613, 911823.621601798, 920137.516688555, 985012.636170723, 905689.866514876, 901634.50606633, 942693.718126975, 924429.103615694, 963618.34963318, 941730.879130773, 966329.717310145, 930737.087270245, 923704.638611525, 958232.298726216, 966203.634976409, 908994.4063453, 990154.526312836, 904149.120557122, 925484.687671997, 968819.946749136, 995816.957787611, 947947.295033373, 953695.721924305, 945483.593270183, 916570.119094104, 911709.597450681, 933188.756695017, 966809.686738998, 976918.964181095, 933472.706354223, 918457.151856273, 983152.1

In [46]:
for col in remaining_columns:
    print("\n" + "=" * 70)
    print("COLUMN:", col)
    
    print("Minimum:", df_clean[col].min())
    print("Median:", df_clean[col].median())
    print("99th percentile:", df_clean[col].quantile(0.99))
    print("Maximum:", df_clean[col].max())


COLUMN: medianmarechprebal30
Minimum: -200.0
Median: 33.9000000000001
99th percentile: 1331.5399999999936
Maximum: 999479.419318959

COLUMN: sumamnt_ma_rech90
Minimum: 0
Median: 7226.0
99th percentile: 78717.23999999996
Maximum: 953036


In [47]:
# Handle clearly invalid values in medianmarechprebal30

col = "medianmarechprebal30"

mask = df_clean[col].abs() >= 900000

print("Values to replace:", mask.sum())

df_clean.loc[mask, col] = np.nan

median_value = df_clean[col].median()

df_clean[col] = df_clean[col].fillna(median_value)

print("Median used for imputation:", median_value)
print("Missing values remaining:", df_clean[col].isna().sum())

Values to replace: 222
Median used for imputation: 33.7399999999998
Missing values remaining: 0


In [48]:
row = df_clean[
    df_clean["sumamnt_ma_rech90"] >= 900000
]

row.T

,26468
label,1
aon,451.0
daily_decr30,47822.832
daily_decr90,48090.24
rental30,-23737.14
rental90,-24720.58
last_rech_date_ma,1.0
last_rech_date_da,0.0
last_rech_amt_ma,10000
cnt_ma_rech30,84


In [49]:
row[
    [
        "label",
        "aon",
        "daily_decr30",
        "daily_decr90",
        "cnt_ma_rech30",
        "sumamnt_ma_rech30",
        "medianamnt_ma_rech30",
        "cnt_ma_rech90",
        "sumamnt_ma_rech90",
        "medianamnt_ma_rech90",
        "medianmarechprebal90",
        "cnt_loans30",
        "amnt_loans30",
        "cnt_loans90",
        "amnt_loans90",
        "payback30",
        "payback90"
    ]
]

,label,aon,daily_decr30,daily_decr90,cnt_ma_rech30,sumamnt_ma_rech30,medianamnt_ma_rech30,cnt_ma_rech90,sumamnt_ma_rech90,medianamnt_ma_rech90,medianmarechprebal90,cnt_loans30,amnt_loans30,cnt_loans90,amnt_loans90,payback30,payback90
26468,1,451.0,47822.832,48090.24,84,810096.0,10000.0,99,953036,10000.0,84.0,1,6,1.0,6,0.0,0.0


In [50]:
print("Shape:", df_clean.shape)

print("\nMissing values:")
print(df_clean.isna().sum().sum())

print("\nDuplicates:")
print(df_clean.duplicated().sum())

print("\nTarget:")
print(df_clean["label"].value_counts())

print("\nDate:")
print(df_clean["pdate"].min(), "to", df_clean["pdate"].max())

Shape: (209593, 34)

Missing values:
0

Duplicates:
31

Target:
label
1    183431
0     26162
Name: count, dtype: int64

Date:
2016-06-01 00:00:00 to 2016-08-21 00:00:00


In [51]:
numeric_cols_clean = df_clean.select_dtypes(
    include=["number"]
).columns

remaining_suspicious = {}

for col in numeric_cols_clean:
    count = (
        df_clean[col].abs() >= 900000
    ).sum()

    if count > 0:
        remaining_suspicious[col] = count

print(
    pd.Series(
        remaining_suspicious,
        name="Remaining_Suspicious_Count"
    )
)

sumamnt_ma_rech90    1
Name: Remaining_Suspicious_Count, dtype: int64


In [52]:
from pathlib import Path

PROCESSED_DATA = Path("../data/processed")
PROCESSED_DATA.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = PROCESSED_DATA / "microfinance_cleaned_base.csv"

df_clean.to_csv(
    OUTPUT_PATH,
    index=False
)

print("Saved:", OUTPUT_PATH)
print("Shape:", df_clean.shape)

Saved: ..\data\processed\microfinance_cleaned_base.csv
Shape: (209593, 34)


In [53]:
saved_df = pd.read_csv(OUTPUT_PATH)

print("File exists:", OUTPUT_PATH.exists())
print("Shape:", saved_df.shape)
print("Missing values:", saved_df.isna().sum().sum())
print("\nTarget distribution:")
print(saved_df["label"].value_counts())

File exists: True
Shape: (209593, 34)
Missing values: 0

Target distribution:
label
1    183431
0     26162
Name: count, dtype: int64
